# PaceForge — Component 1: Data Cleaning

This notebook cleans the synthetic Kaggle marathon dataset for the marathon finish-time
model. The dataset comes as two files:

| File | Raw rows | Output |
|---|---|---|
| `../dataset/train.csv` | 80,000 | `../dataset/clean.csv` |
| `../dataset/test.csv` | 20,000 | `../dataset/clean_test.csv` |

Both files go through the **exact same 6 rules**, written once in a function `clean(df)`.
Every rule is a fixed threshold or a logic check. Nothing is learned from the data (no
means, medians or quantiles), so applying the rules to the test set does not leak
information from the training set.

Scope of this notebook: cleaning only. There is **no imputation** of unknown values,
**no IQR / z-score outlier removal**, and **no encoding or scaling**. (The one fill, in
Step 2, is not imputation: an empty `injury_severity` means "no injury", which we know
exactly from `injury_count`.) Encoding (one-hot for `gender`,
ordinal for `course_difficulty`) and scaling happen later inside a scikit-learn
`Pipeline` in the training notebook, so that nothing leaks from the test set.

In [1]:
import numpy as np
import pandas as pd

TRAIN_PATH = "../dataset/train.csv"
TEST_PATH = "../dataset/test.csv"
TRAIN_OUTPUT = "../dataset/clean.csv"
TEST_OUTPUT = "../dataset/clean_test.csv"

TARGET = "actual_finish_time_minutes"

## Step 1 — Load both files

Load the raw CSVs and confirm the row counts: 80,000 (train) and 20,000 (test).
The test file has 40 columns instead of 41: it has no `medal_outcome` column. This does
not matter, because Step 5 keeps a fixed list of columns instead of dropping columns
by name.

In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("Train shape:", train_raw.shape)
print("Test shape: ", test_raw.shape)
print("Columns in train but not in test:", set(train_raw.columns) - set(test_raw.columns))

assert len(train_raw) == 80_000
assert len(test_raw) == 20_000

Train shape: (80000, 41)
Test shape:  (20000, 40)
Columns in train but not in test: {'medal_outcome'}


## Step 2 — Injury severity: check, then fill

`injury_severity` is empty (NaN) for many runners. In the raw data it is empty **exactly
when** `injury_count == 0`, so an empty value means "this runner had no injury", not
"unknown". We first assert this on both raw files, then fill the empty values with the
label **"No injury"**. The values are then: No injury, Minor, Moderate, Severe.

Why "No injury" and not "None": pandas reads the text "None" back as NaN by default, so
a "None" label saved to CSV would turn back into missing values the next time the file
is loaded.

In [3]:
for name, raw in [("Train", train_raw), ("Test", test_raw)]:
    is_empty = raw["injury_severity"].isna()
    no_injury = raw["injury_count"] == 0
    assert (is_empty == no_injury).all(), f"{name}: empty severity does not match injury_count == 0"
    print(f"{name}: {is_empty.sum()} empty injury_severity values, all with injury_count == 0")

train_raw["injury_severity"] = train_raw["injury_severity"].fillna("No injury")
test_raw["injury_severity"] = test_raw["injury_severity"].fillna("No injury")

print()
print("Test values after filling:")
print(test_raw["injury_severity"].value_counts())

Train: 47350 empty injury_severity values, all with injury_count == 0
Test: 11840 empty injury_severity values, all with injury_count == 0

Test values after filling:
injury_severity
No injury    11840
Minor         4186
Moderate      2535
Severe        1439
Name: count, dtype: int64


## Step 3 — The 6 cleaning rules

All 6 rules live in one function, `clean(df)`, so train and test are cleaned the same way.
The function returns the cleaned data and a small log of counts, which we assert against
the expected values afterwards.

**Rule 1 — Drop rows with a missing target.**
`actual_finish_time_minutes` is the value we want to predict. A row without it cannot be
used for training or testing, so it is dropped (not imputed).

**Rule 2 — Rebuild weekly km from miles (corrupted rows only).**
`weekly_mileage_km` is corrupted in some rows: it shows 100–120 km while
`weekly_mileage_miles` is normal. A row is flagged as corrupted when the two columns
disagree by more than 0.2 km (1 mile = 1.60934 km). In the normal rows the two columns
never differ by more than about 0.08 km, so 0.2 km cleanly separates the two groups.
`weekly_km` keeps the **original km value** in normal rows, and only the corrupted rows
are replaced with `weekly_mileage_miles × 1.60934`. Both original mileage columns are
then dropped.
Some corrupted rows also have a missing target, so they are already gone after Rule 1.
That is why the log records the corrupted count twice: on the raw data and after Rule 1.

**Rule 3 — Target plausibility check (Feely et al., 2023).**
Marathon finish times should fall between 2 and 7 hours, which is 120–420 minutes. This is
a **check, not a filter**: the function asserts that every value is inside the range and
removes nothing.

**Rule 4 — Remove BMI ≥ 30.**
Following the NURMI study, which uses the WHO obesity criterion, runners with BMI ≥ 30
are removed.

**Rule 5 — Remove long run longer than weekly km.**
A runner's single long run cannot be longer than their total weekly distance. These rows
are logically impossible. The rule uses the rebuilt `weekly_km` from Rule 2.

**Rule 6 — Remove running experience longer than the runner's age.**
`running_experience_months` cannot be more than the runner's age in months (`age × 12`).

In [4]:
def clean(df):
    log = {"raw": len(df)}

    # Count corrupted mileage rows in the raw data (diagnostic only)
    raw_bad = (df.weekly_mileage_km - df.weekly_mileage_miles * 1.60934).abs() > 0.2
    log["raw_bad_km"] = raw_bad.sum()

    # Rule 1: drop rows with a missing target
    df = df[df[TARGET].notna()].copy()
    log["after_target"] = len(df)

    # Rule 2: rebuild weekly km, replacing only the corrupted rows
    bad = (df.weekly_mileage_km - df.weekly_mileage_miles * 1.60934).abs() > 0.2
    log["bad_km"] = bad.sum()
    df["weekly_km"] = np.where(bad, df.weekly_mileage_miles * 1.60934, df.weekly_mileage_km)
    df = df.drop(columns=["weekly_mileage_km", "weekly_mileage_miles"])

    # Rule 3: target plausibility check (120-420 min); a check, not a filter
    log["min_time"] = df[TARGET].min()
    log["max_time"] = df[TARGET].max()
    assert df[TARGET].between(120, 420).all(), "Finish time outside 120-420 min"

    # Rule 4: remove BMI >= 30
    df = df[df["bmi"] < 30]
    log["after_bmi"] = len(df)

    # Rule 5: remove long run longer than weekly km
    df = df[df["long_run_distance_km"] <= df["weekly_km"]]
    log["after_long_run"] = len(df)

    # Rule 6: remove running experience longer than age in months
    df = df[df["running_experience_months"] <= df["age"] * 12]
    log["after_experience"] = len(df)

    return df, log

## Step 4 — Clean both files and check the counts

The expected counts below come from a verified check of each file. Every value the
function logs is asserted against them.

In [5]:
EXPECTED_TRAIN = {
    "raw": 80_000,
    "raw_bad_km": 2_143,
    "after_target": 78_011,
    "bad_km": 1_980,
    "min_time": 171,
    "max_time": 400,
    "after_bmi": 77_801,
    "after_long_run": 77_589,
    "after_experience": 77_577,
}

EXPECTED_TEST = {
    "raw": 20_000,
    "raw_bad_km": 538,
    "after_target": 19_489,
    "bad_km": 483,
    "min_time": 169,
    "max_time": 398,
    "after_bmi": 19_434,
    "after_long_run": 19_382,
    "after_experience": 19_380,
}

train, train_log = clean(train_raw)
test, test_log = clean(test_raw)

for name in EXPECTED_TRAIN:
    assert train_log[name] == EXPECTED_TRAIN[name], f"Train {name}: got {train_log[name]}"
    assert test_log[name] == EXPECTED_TEST[name], f"Test {name}: got {test_log[name]}"

print("All train and test counts match the expected values.")

All train and test counts match the expected values.


## Step 5 — Keep only the needed columns and save

| Role | Columns |
|---|---|
| ID | `runner_id` |
| 9 model features | `age`, `gender`, `bmi`, `weekly_km`, `runs_per_week`, `long_run_distance_km`, `course_difficulty`, `resting_heart_rate_bpm`, `running_experience_months` |
| Target | `actual_finish_time_minutes` |
| Evaluation only (not a feature) | `personal_best_minutes` |
| Injury history | `injury_severity` (No injury / Minor / Moderate / Severe) |

`injury_severity` is kept in the cleaned files but is **not** one of the 9 model
features used by the training notebook.

We **select** these 13 columns instead of dropping the others by name. This is safe for
the test file, which has no `medal_outcome` column, and both outputs end up with the
same columns in the same order.

`personal_best_minutes` is kept only for the Riegel baseline and the PB ablation later.
It is empty (NaN) for first-time marathoners, which is correct, so it is **not** filled.

`target_finish_time_minutes` and `medal_outcome` are **not** kept because they leak
information about the result.

In [6]:
FEATURES = [
    "age",
    "gender",
    "bmi",
    "weekly_km",
    "runs_per_week",
    "long_run_distance_km",
    "course_difficulty",
    "resting_heart_rate_bpm",
    "running_experience_months",
]
EVAL_ONLY = "personal_best_minutes"
INJURY = "injury_severity"
KEEP_COLUMNS = ["runner_id"] + FEATURES + [TARGET, EVAL_ONLY, INJURY]

train = train[KEEP_COLUMNS]
test = test[KEEP_COLUMNS]

train.to_csv(TRAIN_OUTPUT, index=False)
test.to_csv(TEST_OUTPUT, index=False)

print("Saved", TRAIN_OUTPUT, "with shape", train.shape)
print("Saved", TEST_OUTPUT, "with shape", test.shape)
print("personal_best_minutes missing (first-time marathoners): train",
      train[EVAL_ONLY].isna().sum(), "| test", test[EVAL_ONLY].isna().sum())

Saved ../dataset/clean.csv with shape (77577, 13)
Saved ../dataset/clean_test.csv with shape (19380, 13)
personal_best_minutes missing (first-time marathoners): train 10415 | test 2571


## Step 6 — Final checks

Confirm the final row counts, that both files have the same 13 columns, and that the 9
features, the target and `injury_severity` have no missing values. For `injury_severity`
we also re-read the saved `clean_test.csv` from disk, to prove the "No injury" label
survives saving and loading, and check that "No injury" is the largest group. Then show
the row count after every step for both files.

In [7]:
assert len(train) == 77_577
assert len(test) == 19_380
assert train.shape[1] == 13
assert list(train.columns) == list(test.columns)

assert train[FEATURES + [TARGET, INJURY]].isna().sum().sum() == 0
assert test[FEATURES + [TARGET, INJURY]].isna().sum().sum() == 0
print("No missing values in the features, the target or injury_severity (train and test).")

saved_test = pd.read_csv(TEST_OUTPUT)
injury_counts = saved_test[INJURY].value_counts(dropna=False)
print()
print("injury_severity in the saved clean_test.csv:")
print(injury_counts)
assert saved_test[INJURY].isna().sum() == 0
assert set(injury_counts.index) == {"No injury", "Minor", "Moderate", "Severe"}
assert injury_counts.idxmax() == "No injury"
print()

steps = [
    ("Load raw data", "raw"),
    ("Rule 1: drop missing target", "after_target"),
    ("Rule 2: rebuild weekly_km", "after_target"),
    ("Rule 3: target check (120-420 min)", "after_target"),
    ("Rule 4: remove BMI >= 30", "after_bmi"),
    ("Rule 5: remove long run > weekly km", "after_long_run"),
    ("Rule 6: remove experience > age x 12", "after_experience"),
]
summary = pd.DataFrame({
    "Step": [step for step, key in steps],
    "Train rows": [train_log[key] for step, key in steps],
    "Test rows": [test_log[key] for step, key in steps],
})
summary["Train dropped"] = (summary["Train rows"].shift(1) - summary["Train rows"]).fillna(0).astype(int)
summary["Test dropped"] = (summary["Test rows"].shift(1) - summary["Test rows"]).fillna(0).astype(int)

print()
print(f"Corrupted km rows fixed by Rule 2: train {train_log['bad_km']} "
      f"(raw {train_log['raw_bad_km']}), test {test_log['bad_km']} (raw {test_log['raw_bad_km']})")
print(f"Finish time range: train {train_log['min_time']}-{train_log['max_time']} min, "
      f"test {test_log['min_time']}-{test_log['max_time']} min")
summary

No missing values in the features, the target or injury_severity (train and test).

injury_severity in the saved clean_test.csv:
injury_severity
No injury    11644
Minor         4052
Moderate      2365
Severe        1319
Name: count, dtype: int64


Corrupted km rows fixed by Rule 2: train 1980 (raw 2143), test 483 (raw 538)
Finish time range: train 171.0-400.0 min, test 169.0-398.0 min


,Step,Train rows,Test rows,Train dropped,Test dropped
0,Load raw data,80000,20000,0,0
1,Rule 1: drop missing target,78011,19489,1989,511
2,Rule 2: rebuild weekly_km,78011,19489,0,0
3,Rule 3: target check (120-420 min),78011,19489,0,0
4,Rule 4: remove BMI >= 30,77801,19434,210,55
5,Rule 5: remove long run > weekly km,77589,19382,212,52
6,Rule 6: remove experience > age x 12,77577,19380,12,2


## Step 7 — Data quality report

These numbers describe weaknesses of the synthetic dataset, for the Limitations section.
They are computed on the cleaned training set and are reported only; nothing is changed.

In [8]:
pct_low_mileage = (train["weekly_km"] <= 20.0).mean() * 100
pct_long_run_15 = (train["long_run_distance_km"] == 15.0).mean() * 100
pct_rhr_80 = (train["resting_heart_rate_bpm"] == 80).mean() * 100

print(f"Rows with weekly_km <= 20.0:            {pct_low_mileage:.1f}%")
print(f"Rows with long_run_distance_km == 15.0: {pct_long_run_15:.1f}%")
print(f"Rows with resting_heart_rate_bpm == 80: {pct_rhr_80:.1f}%")
print()
print("Mean finish time by gender (minutes):")
print(train.groupby("gender")[TARGET].agg(["count", "mean"]).round(1))

Rows with weekly_km <= 20.0:            49.9%
Rows with long_run_distance_km == 15.0: 81.0%
Rows with resting_heart_rate_bpm == 80: 40.0%

Mean finish time by gender (minutes):
            count   mean
gender                  
Female      37486  287.9
Male        37053  287.9
Non-binary   3038  288.7
